# Ranking Concepts

**Module:** 03 — Reranking

Relevance labels, offline metrics, and learning-to-rank styles are the language of search quality. Master them before tuning rerankers.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define graded relevance and label noise realities
- Compute and interpret Precision@k, Recall@k, MRR, nDCG
- Contrast pointwise, pairwise, and listwise LTR
- Separate offline metrics from online product metrics
- Build a tiny eval harness for reranker experiments


## Relevance

**Definition.** **Relevance** is how well a document satisfies an information need for a query—often graded (0/1/2/3) rather than binary.

**Why it matters.** Without a relevance definition, 'better ranking' is undefined and A/Bs thrash.

**How it works.** Write labeling guidelines; sample queries; double-label a slice; adjudicate disagreements.

**Intuition.** Relevance is a job interview score for each doc given the question—not a vibe.

**Common pitfalls.**
- Binary labels when partial answers matter
- Labeling on titles only while models see full text
- Leaking future clicks into training labels carelessly
- One annotator, no agreement check

**When to use.** Before any model bakeoff—define the scale and guidelines.

### Graded relevance cheat sheet

| Grade | Meaning | Typical use in RAG |
|------:|---------|--------------------|
| 3 | Answers the question | Must be in top-k if present |
| 2 | Partial / supporting | Good context |
| 1 | Related topic | Optional |
| 0 | Irrelevant | Should rank low |


In [ ]:
# Demo 1 — graded relevance example
labels = {
    "q:refund": {"policy.md#0": 3, "shipping.md#0": 0, "returns.md#1": 2, "blog.md#4": 1},
}
for q, docs in labels.items():
    print(q)
    for d, g in sorted(docs.items(), key=lambda x: -x[1]):
        print(f"  grade={g} {d}")


In [ ]:
# Demo 2 — inter-annotator disagreement
a = {"d1": 3, "d2": 1, "d3": 0}
b = {"d1": 2, "d2": 2, "d3": 0}
agree = sum(a[k] == b[k] for k in a) / len(a)
print("exact agreement", agree)
print("near agreement (±1)", sum(abs(a[k]-b[k]) <= 1 for k in a) / len(a))


In [ ]:
# Demo 3 — query intent types affect relevance
intents = {
    "navigational": "find a specific page",
    "informational": "learn a fact/procedure",
    "transactional": "complete an action",
}
for k, v in intents.items():
    print(f"{k:15s} {v}")


### Try it yourself — Relevance

1. Write a 4-level grade rubric for an internal HR policy search.
2. Give an example where two annotators reasonably disagree.


## Metrics

**Definition.** **Ranking metrics** score an ordered list against graded/binary judgments—Precision@k, Recall@k, MRR, MAP, nDCG@k are the workhorses.

**Why it matters.** You cannot improve what you do not measure; metrics catch silent regressions when you change embedders, N, or rerankers.

**How it works.** For each query, take the ranked IDs, compare to labels, average over the eval set (macro). Report confidence intervals when possible.

**Intuition.** nDCG rewards putting highly graded docs early; MRR cares about the first relevant hit.

**Common pitfalls.**
- Optimizing Precision@1 when users need diverse evidence in top-5
- Huge eval sets that are never refreshed
- Mixing train queries into the test metric
- Reporting only mean without hard-query slices

**When to use.** Use nDCG@k for graded RAG shortlists; MRR for 'first correct link' UIs.

### Offline vs online

| Offline | Online |
|---------|--------|
| Labeled sets, reproducible | User behavior, noisy |
| Good for model bakeoffs | Good for product impact |
| Can overfit to guidelines | Needs careful experiments |


In [ ]:
# Demo 1 — Precision@k and Recall@k
def precision_at_k(ranked, relevant, k):
    top = ranked[:k]
    return len(set(top) & set(relevant)) / k

def recall_at_k(ranked, relevant, k):
    if not relevant:
        return 0.0
    return len(set(ranked[:k]) & set(relevant)) / len(relevant)

ranked = ["d2", "d1", "d9", "d3"]
relevant = {"d1", "d3"}
print("P@2", precision_at_k(ranked, relevant, 2))
print("R@3", recall_at_k(ranked, relevant, 3))


In [ ]:
# Demo 2 — MRR
def mrr(ranked, relevant):
    for i, doc in enumerate(ranked, 1):
        if doc in relevant:
            return 1.0 / i
    return 0.0

print("MRR", mrr(["x", "d1", "y"], {"d1"}))


In [ ]:
# Demo 3 — nDCG@k
import math

def dcg(rels):
    return sum((2**r - 1) / math.log2(i + 2) for i, r in enumerate(rels))

def ndcg_at_k(ranked_grades, k):
    gains = ranked_grades[:k]
    ideal = sorted(ranked_grades, reverse=True)[:k]
    idcg = dcg(ideal)
    return 0.0 if idcg == 0 else dcg(gains) / idcg

print("nDCG@3", round(ndcg_at_k([3, 0, 2, 1], 3), 4))
print("nDCG@3 ideal-ish", round(ndcg_at_k([3, 2, 1, 0], 3), 4))


In [ ]:
# Demo 4 — offline vs online metrics
offline = ["nDCG@10", "MRR", "Recall@50"]
online = ["click@1", "resolve_rate", "thumbs_up", "p95_latency"]
print("offline:", offline)
print("online:", online)


### Try it yourself — Metrics

1. Compute nDCG@3 by hand for grades [2, 3, 0].
2. Why can Recall@50 be high while RAG still fails?


## Learning to Rank Styles

**Definition.** **Learning to Rank (LTR)** trains models to order documents. Losses are **pointwise** (predict grade), **pairwise** (prefer A over B), or **listwise** (optimize list metrics).

**Why it matters.** Cross-encoder rerankers are often trained with pairwise/listwise objectives on MS MARCO-style pairs; understanding the loss explains failure modes.

**How it works.** Collect (query, doc, label) or preference pairs → featurize or use a Transformer → minimize the chosen loss → evaluate with nDCG.

**Intuition.** Pointwise grades each student alone; pairwise asks who ranks higher; listwise grades the whole class order.

**Common pitfalls.**
- Pairwise training with massively imbalanced negatives
- Optimizing pointwise MSE when you only care about top-5 order
- Leaking rank position features that won't exist at serve time

**When to use.** Choose pairwise/listwise for rerankers; pointwise for simple classifiers/filters.

```mermaid
flowchart TB
  subgraph styles [LTR styles]
    PW[Pointwise]
    PR[Pairwise]
    LW[Listwise]
  end
  PW --> M[Rerank model]
  PR --> M
  LW --> M
  M --> E[nDCG / MRR]
```


In [ ]:
# Demo 1 — pointwise vs pairwise targets
print("pointwise: predict grade g in {0,1,2,3}")
print("pairwise:  given (d+, d-), score(d+) > score(d-)")
print("listwise:  soft distribution over permutations / metric proxies")


In [ ]:
# Demo 2 — pairwise hinge on toy scores
def hinge(s_pos, s_neg, margin=1.0):
    return max(0.0, margin - (s_pos - s_neg))

print("good", hinge(2.5, 0.5))
print("bad ", hinge(0.2, 1.1))


In [ ]:
# Demo 3 — listwise softmax CE sketch
import numpy as np
scores = np.array([2.0, 0.5, 0.1])
grades = np.array([3.0, 1.0, 0.0])
p = np.exp(scores - scores.max()); p /= p.sum()
q = np.exp(grades - grades.max()); q /= q.sum()
print("model", p.round(3), "target", q.round(3))


In [ ]:
# Demo 4 — feature LTR vs cross-encoder
rows = [
    ("GBDT LTR", "hand features + clicks", "fast CPU"),
    ("bi-encoder", "dual towers", "retrieval scale"),
    ("cross-encoder", "joint Transformer", "rerank shortlist"),
]
for name, sig, note in rows:
    print(f"{name:14s} | {sig:22s} | {note}")


### Try it yourself — Learning to Rank Styles

1. Map MS MARCO triplet training to pairwise loss in one sentence.
2. When would a GBDT LTR still beat a CE on tabular/site search?


## Glossary

- **nDCG**: Normalized Discounted Cumulative Gain
- **MRR**: Mean Reciprocal Rank
- **pointwise**: Predict a relevance score per doc
- **pairwise**: Learn relative order between pairs


### Workshop drill — Ranking Concepts (1)

Restate each major section heading as one exam-ready sentence.


In [ ]:
# Workshop drill 1 — Ranking Concepts
headings = ['Relevance', 'Metrics']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Ranking Concepts (2)

Sketch a latency budget: first-stage ms + rerank (N candidates × cost) + LLM.


In [ ]:
# Workshop drill 2 — Ranking Concepts
first_ms, per_pair_ms, n, llm_ms = 40, 3, 50, 800
print('total_ms', first_ms + n*per_pair_ms + llm_ms)
print('rerank_share', round(n*per_pair_ms/(first_ms+n*per_pair_ms+llm_ms), 3))


### Workshop drill — Ranking Concepts (3)

Design an offline metric slice: 5 queries with graded relevance labels.


In [ ]:
# Workshop drill 3 — Ranking Concepts
eval_set = [{'q':'...','docs':{'d1':2,'d2':1,'d3':0}}]
print('n_queries', len(eval_set))
print('TODO: fill real labels')


### Workshop drill — Ranking Concepts (4)

Write a go/no-go checklist for shipping a reranker in RAG.


In [ ]:
# Workshop drill 4 — Ranking Concepts
for c in ['latency_p95','nDCG@10','cost/1k','cache_hit','fallback']:
    print(f'[ ] {c}')


## Summary & Key Takeaways

- Define graded relevance before tuning models
- nDCG@k fits graded shortlists; MRR fits first-hit UIs
- Pairwise/listwise losses match ranking goals better than naive regression
- Offline gates + online confirmation is the healthy loop

### Practice

Build a 10-query graded set and compute nDCG@5 for two artificial rankings.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
